In [ ]:
from pathlib import Path
import getpass
import os
import re

# Works when this notebook starts in notebooks/, the repository root, or Gadi.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))]
repo_root = next((path for path in candidates
                  if (path / "exercises/CMakeLists.txt").is_file()
                  and (path / "job-script").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the intro-to-cuda checkout.")
os.chdir(repo_root)
job_dir = repo_root / "job-script"
username = getpass.getuser()


def show_job_output(job_name, job_id):
    job_number = job_id.split(".")[0]
    log = job_dir / f"{job_name}.o{job_number}"
    if log.is_file():
        print(log.read_text())
    else:
        print(f"Output is not available yet: {log.name}. Check qstat, then rerun this cell.")


print(f"Repository: {repo_root}")

def show_sample_output(filename, label="Saved sample output"):
    log = repo_root / "sample-outputs" / filename
    text = log.read_text()
    print(f"{label} ({log.name})\n")
    print(text, end="")
    return text


# Shared Memory in CUDA


> **Overview**
>
> 
>  * **Time:** 10 min
> 
>  1. Learn about shared memory in CUDA.
>  1. Understand the characteristics and usage of shared memory.

Shared memory is a type of memory that is shared among threads within the same block in CUDA. It is much
faster than global memory and can be used to store data that needs to be accessed by multiple threads.
Shared memory is particularly useful for reducing global memory accesses and improving performance in
parallel algorithms.

## Shared Memory Characteristics

Shared memory in CUDA has several key characteristics:

1. **Scope**: Shared memory is accessible only to threads within the same block. Threads in different blocks cannot access each other's shared memory.

2. **Speed**: Shared memory is on-chip memory, making it much faster than global memory (DRAM). Accessing shared memory typically has a latency of around 10-50 cycles, compared to 400-800 cycles for global memory.

3. **Size**: The size of shared memory is limited, typically ranging from 48 KB to 96 KB per block, depending on the GPU architecture.

4. **Coherency**: Shared memory is coherent across all threads in a block, meaning that changes made by one thread are visible to others in the same block.

5. **Usage**: Shared memory is often used for data that needs to be accessed frequently by multiple threads, such as intermediate results in parallel computations.

**Shared Memory vs DRAM in NVIDIA GPUs**

| Feature | Shared Memory | DRAM (Global Memory) |
| --- | --- | --- |
| Access speed | Very fast (10–50 cycles) | Slow (400–800 cycles) |
| Capacity | Small (48–96 KB per SM) | Large (GBs) |
| Usage | Frequently accessed data within a block | Data shared across all blocks |
| Benefit | Reduces global memory traffic and accelerates access | Higher latency, but larger capacity |

There are two main types of shared memory in CUDA:

1. **Static Shared Memory**: This is defined at compile time and has a fixed size. It is declared using the `__shared__` keyword in the kernel code. Static shared memory is allocated for each block and is initialized when the block is launched.

```cpp
__global__ void addKernel(int *input, int *output) 
{
    __shared__ int sdata[256];  // static shared memory

    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Load data from global to shared memory
    sdata[tid] = input[idx];
    __syncthreads();

    // Do something with shared memory
    sdata[tid] += 10;
    __syncthreads();

    // Store result back to global memory
    output[idx] = sdata[tid];
}
```

2. **Dynamic Shared Memory**: This is allocated at runtime and can vary in size based on the needs of the kernel. It is declared using the `extern __shared__` keyword in the kernel code. Dynamic shared memory allows for more flexibility, as its size can be specified when launching the kernel.

```cpp
__global__ void addKernel(int *input, int *output) 
{
    extern __shared__ int sdata[];  // dynamic shared memory
    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Load data from global to shared memory
    sdata[tid] = input[idx];
    __syncthreads();

    // Do something with shared memory
    sdata[tid] += 10;
    __syncthreads();

    // Store result back to global memory
    output[idx] = sdata[tid];
}

int main() 
{
    int threads = 256;
    int sharedMemSize = threads * sizeof(int);  // Size of dynamic shared memory

    ...
    ...
    ...

    addKernel<<<1, threads, sharedMemSize>>>(d_input, d_output);
}
```

> **Important**
>
> You declare only one `extern __shared__ array` inside your kernel.

> **Explanation**
>
> 
>  `__syncthreads()` is a barrier synchronization function in CUDA that ensures all threads in the same 
>  thread block:
> 
>  1. Reach the barrier, and
>  2. Complete all memory accesses (reads/writes to shared memory)
> 
>  before any thread in the block continues past it.

> **Key Points**
>
> 
>  1. Shared memory is a fast, on-chip memory accessible by threads within the same block.
>  1. It is used to reduce global memory accesses and improve performance in parallel algorithms.
>  1. There are two types of shared memory: static (fixed size) and dynamic (runtime size).
>  1. The `__syncthreads()` function is used to synchronize threads within a block, ensuring all threads reach a certain point before proceeding.

## Working with the examples

The CUDA code above illustrates declarations and launch syntax. These are fragments: the dynamic-memory example uses `...` for omitted host setup and cannot be compiled as written. Both kernels assume their input/output buffers cover every launched thread; the static example also assumes at most 256 threads per block.

The first kernel uses a fixed-size shared array; the second receives the shared-memory size in bytes through the third launch argument. Threads must write the required values before reading them; shared arrays are not automatically zero-initialized.

## Exercise 07: Shared Memory

**Time: 10 min**

[Open swap-neighbours.cu](../exercises/07-shared-memory/swap-neighbours.cu).

One block of four threads writes 10, 20, 30, and 40 into a shared array.
After `__syncthreads()`, each thread reads the next slot, with thread 3
wrapping around to slot 0. Predict the four values in thread order.

### Submit the exercise

Run the setup cell first. Build the exercises in [02-gpu.ipynb](02-gpu.ipynb),
then submit the job. Rebuild after editing the CUDA source.

In [ ]:
# Submit a new job, then inspect the queue.
submission = !cd "{job_dir}" && qsub 07-shared-memory.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


### Check the GPU output

Run this cell to check the queue and read the submitted job's log. If the job
is still queued or running, rerun this cell after it finishes.

The `Read` column should contain 20, 30, 40, and 10. Explain how the barrier
makes each neighbour's write available before the read.

In [ ]:
!qstat -u "{username}"
show_job_output("07-shared-memory", job_id)


### Use saved output while the job is queued

Run this cell after setup to see the expected neighbour exchange. The archived PBS log in sample-outputs is from the earlier brightness exercise.

In [ ]:
_ = show_sample_output("07-shared-memory-expected.txt", label="Expected output for the neighbour exchange")